# Analise de Estabilidade do Controlador Fuzzy PDC (LMI)

Notebook dedicado **exclusivamente** a verificar a estabilidade dos controladores otimizados (GA e QGA), **sem re-rodar a otimizacao**. Usa os polos otimos ja salvos em `maglev_complete_optimization_data_32bits.pkl`.

**Metodo:**
1. Para cada ponto de operacao, reconstroi o modelo linearizado aumentado (estados $x_1, x_2, x_3, x_I$).
2. Recalcula os ganhos $K_i$ por alocacao de polos a partir dos polos otimos salvos (consistencia: os autovalores da malha fechada coincidem com os polos).
3. **Estabilidade local:** verifica se cada malha fechada local e Hurwitz.
4. **Estabilidade quadratica (Tanaka & Wang):** procura uma matriz de Lyapunov $P \succ 0$ **comum** satisfazendo, para as regras ativas, os termos diagonais e cruzados adjacentes. Testa diferentes faixas de operacao.

**Observacao numerica:** as matrizes possuem entradas grandes (o termo de instabilidade chega a $\sim 2\times10^4$). Por isso aplica-se uma mudanca de coordenadas diagonal (similaridade comum), que nao altera a conclusao de estabilidade, apenas o condicionamento do SDP.

In [ ]:
import pickle
from pathlib import Path

import numpy as np
from scipy.signal import place_poles
import cvxpy as cp
print('cvxpy', cp.__version__, '| solvers SDP:', [s for s in ['CLARABEL','SCS'] if s in cp.installed_solvers()])
# PT-BR: Se o CVXPY não estiver instalado, execute: %pip install cvxpy
# EN-US: If CVXPY is not installed, run: %pip install cvxpy

In [ ]:
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = Path.cwd().resolve().parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Execute este notebook dentro do repositório.")

ARQ = REPO_ROOT / "data" / "maglev_complete_optimization_data_32bits.pkl"
with ARQ.open("rb") as arquivo:
    d = pickle.load(arquivo)
pm = d['config_info']['original']['sistema_maglev_params']
C, m, R, L = pm['K_m'], pm['M_b'], pm['R_c'], pm['L_c']
ops = np.array(d['config_info']['original']['pontos_operacao'])  # PT-BR: colunas: x_b0 [m], i_c0 [A]
# EN-US: columns: x_b0 [m], i_c0 [A]
mm = ops[:, 0]*1000.0
# PT-BR: Escala diagonal (posição em mm etc.) para condicionar o SDP. A similaridade comum não altera a conclusão.
# EN-US: Diagonal scaling (position in mm, etc.) conditions the SDP. The common similarity transformation does not change the conclusion.
D = np.diag([1e3, 1e1, 1.0, 1e1]); Di = np.linalg.inv(D)
print('Parametros:', pm)
print('Pontos de operacao (mm):', mm.astype(int))

In [ ]:
def sistema_aumentado(x1, x3):
    '''PT-BR: Modelo linearizado aumentado (x1, x2, x3, xI) no ponto (x1, x3).

    EN-US: Augmented linearized model (x1, x2, x3, xI) at point (x1, x3).'''
    a21 = 2.0*(C/m)*x3**2/x1**3
    a23 = -2.0*(C/m)*x3/x1**2
    A = np.array([[0.0,1.0,0.0],[a21,0.0,a23],[0.0,0.0,-R/L]])
    B = np.array([[0.0],[0.0],[1.0/L]])
    Aaug = np.block([[A, np.zeros((3,1))],[np.array([[-1.0,0.0,0.0]]), np.array([[0.0]])]])
    Baug = np.vstack([B, [[0.0]]])
    return Aaug, Baug

def polos_do_vetor(v):
    v = np.array(v)
    if np.any(np.abs(np.imag(v)) > 1e-9):
        return v.astype(complex)
    a, b, c, e = [float(x) for x in v]
    return np.array([-a, -b, -c+1j*e, -c-1j*e])

def montar(polos_vec):
    '''PT-BR: Retorna A_i, B_i e K_i, já escaladas por D, além dos polos.

    EN-US: Return A_i, B_i, and K_i, already scaled by D, and the poles.'''
    polos = polos_do_vetor(polos_vec)
    AL, BL, KL = [], [], []
    for (x1, x3) in ops:
        Aa, Ba = sistema_aumentado(x1, x3)
        K = place_poles(Aa, Ba, polos).gain_matrix
        AL.append(D @ Aa @ Di); BL.append(D @ Ba); KL.append(K @ Di)
    return AL, BL, KL, polos

In [ ]:
def hurwitz_todos(AL, BL, KL):
    '''PT-BR: Verifica se toda malha fechada A_i - B_i K_i é Hurwitz.

    EN-US: Check whether every closed-loop A_i - B_i K_i is Hurwitz.'''
    max_re = -np.inf
    for A, B, K in zip(AL, BL, KL):
        max_re = max(max_re, float(np.max(np.real(np.linalg.eigvals(A - B @ K)))))
    return (max_re < 0.0), max_re

def common_P(AL, BL, KL, idx, cross=True, tol=1e-6):
    '''PT-BR: Verifica estabilidade quadrática (Tanaka-Wang) com P > 0 comum
    aos índices em idx. Inclui termos diagonais e, se cross=True, termos
    cruzados de regras adjacentes.

    EN-US: Check quadratic stability (Tanaka-Wang) with a common P > 0
    over the indices in idx. Include diagonal terms and, when cross=True,
    cross terms for adjacent rules.'''
    A = [AL[i] for i in idx]; B = [BL[i] for i in idx]; K = [KL[i] for i in idx]
    n = 4; P = cp.Variable((n, n), symmetric=True); cons = [P >> np.eye(n)]
    CL = [a - b @ k for a, b, k in zip(A, B, K)]
    for Acl in CL:
        cons.append(Acl.T @ P + P @ Acl << -tol*np.eye(n))
    if cross:
        for i in range(len(A)-1):
            H = ((A[i]-B[i]@K[i+1]) + (A[i+1]-B[i+1]@K[i]))/2.0
            cons.append(H.T @ P + P @ H << 0)
    prob = cp.Problem(cp.Minimize(0), cons)
    try:
        prob.solve(solver=cp.CLARABEL)
    except Exception:
        prob.solve(solver=cp.SCS, eps=1e-7, max_iters=200000)
    return prob.status in ('optimal', 'optimal_inaccurate')

In [ ]:
configs = {
    'IT2-FLC + QGA': d['qga_melhores_polos_qga_t2i'],
    'IT2-FLC + GA' : d['ga_best_poles_T2I_chromosome'],
    'FT1 + QGA'    : d['qga_melhores_polos_qga_t1'],
    'FT1 + GA'     : d['ga_best_poles_T1_chromosome'],
    'Original'     : d['config_info']['original']['polos_desejados'],
}
# PT-BR: Faixas de operação, definidas pela posição em mm.
# EN-US: Operating ranges, defined by position in mm.
def idx_mm(lo, hi):
    return [i for i in range(len(mm)) if lo-0.1 <= mm[i] <= hi+0.1]
faixas = {'2-12mm (grade)': idx_mm(2,12), '3-9mm (trajetoria)': idx_mm(3,9),
          '4-8mm (central)': idx_mm(4,8), '5-9mm (treino)': idx_mm(5,9)}

print('ESTABILIDADE LOCAL (todas as 11 malhas Hurwitz):')
for nome, pv in configs.items():
    AL, BL, KL, polos = montar(pv)
    ok, mr = hurwitz_todos(AL, BL, KL)
    print(f'  {nome:16s}: {ok}  (max Re = {mr:.3f}) | polos {np.round(polos,2)}')

In [ ]:
print('ESTABILIDADE QUADRATICA (P>0 comum, Tanaka-Wang com cruzados adjacentes):')
cab = 'Config'.ljust(16) + ' | ' + ' | '.join(f.ljust(18) for f in faixas)
print(cab); print('-'*len(cab))
for nome, pv in configs.items():
    AL, BL, KL, _ = montar(pv)
    cels = []
    for f, idx in faixas.items():
        viavel = common_P(AL, BL, KL, idx, cross=True)
        cels.append(('SIM' if viavel else 'nao').ljust(18))
    print(nome.ljust(16) + ' | ' + ' | '.join(cels))

In [ ]:
# PT-BR: Ponto de quebra: largura a partir da qual deixa de existir um Lyapunov quadrático comum para IT2-FLC + QGA.
# EN-US: Breakpoint: range width beyond which a common quadratic Lyapunov function no longer exists for IT2-FLC + QGA.
AL, BL, KL, _ = montar(configs['IT2-FLC + QGA'])
print('IT2-FLC + QGA  | common-P incluindo pontos 2mm..(2+k):')
for k in range(len(mm)):
    idx = list(range(0, k+1))
    print(f'  ate {mm[k]:.0f}mm ({k+1} pts): {"SIM" if common_P(AL,BL,KL,idx,cross=True) else "nao"}')

In [ ]:
# PT-BR: Varredura de janelas [lo, hi]: compara a região certificável do
# controlador original com a do IT2-FLC + QGA.
# PT-BR: Resultado esperado, conforme execução de 09/07/2026:
# - o conjunto de janelas certificáveis do original contém estritamente o
#   do QGA (o original certifica 2-6, 3-9 e 3-10; o QGA não);
# - a largura máxima certificável é igual para ambos: 8 pontos, em 4-11 ou 5-12;
# - a diferença está na posição: o QGA perde a certificação na região inferior
#   (2-3 mm), próxima ao eletroímã, onde a não linearidade é mais severa.
# EN-US: Window sweep [lo, hi]: compare the certifiable region of the
# original controller with that of IT2-FLC + QGA.
# EN-US: Expected result, based on the run performed on July 9, 2026:
# - the original controller's set of certifiable windows strictly contains
#   the QGA set (the original certifies 2-6, 3-9, and 3-10; QGA does not);
# - the maximum certifiable width is the same for both: 8 points, at 4-11 or 5-12;
# - the difference lies in position: QGA loses certification in the lower
#   region (2-3 mm), near the electromagnet, where nonlinearity is more severe.
AL_o, BL_o, KL_o, _ = montar(configs['Original'])
AL_q, BL_q, KL_q, _ = montar(configs['IT2-FLC + QGA'])
print('Janela      Original  IT2-QGA')
for lo in range(2, 7):
    for hi in range(lo+4, 13):
        idx = idx_mm(lo, hi)
        so = 'SIM' if common_P(AL_o, BL_o, KL_o, idx, cross=True) else 'nao'
        sq = 'SIM' if common_P(AL_q, BL_q, KL_q, idx, cross=True) else 'nao'
        print(f'{lo:>2}-{hi:<2}mm     {so:9s} {sq}')

## Interpretação e limites

- As 11 malhas locais são Hurwitz para todas as configurações avaliadas.
- A tabela de faixas selecionadas encontrou uma matriz de Lyapunov quadrática
  comum em 4--8 mm e 5--9 mm para todas as configurações; em 3--9 mm, apenas
  o controlador original foi certificado. Nenhuma configuração obteve
  certificado comum em 2--12 mm.
- A varredura adicional certificou, entre outras, a janela de 4--9 mm para o
  IT2-FLC + QGA. O conjunto de janelas certificadas do controlador original
  contém estritamente o do otimizado; a largura máxima observada foi de oito
  pontos para ambos.
- A análise é *a posteriori*, condicionada à grade, aos ganhos arquivados, às
  LMIs formuladas e ao solver. O estado `optimal_inaccurate`, quando retornado,
  deve ser interpretado com cautela.

Referência metodológica: K. Tanaka e H. O. Wang, *Fuzzy Control Systems Design
and Analysis: A Linear Matrix Inequality Approach*, Wiley, 2001.
